In [18]:
#[CELL 1] STUDENT CONFIGURATION
import csv, json, shutil
from pathlib import Path

LAST_NAME = "Dantes"
STUDENT_ID = "TUPM-26-7955"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_CLASS = (SEED_DIGIT % 5) + 1

EXPORT_FIELDS = ["User ID", "Device Model", "Operating System",
                 "Battery Drain (mAh/day)", "Data Usage (MB/day)"]

print("Seed:", SEED_DIGIT)
print("Target operating system:", TARGET_OS)
print("Target user behavior class:", TARGET_CLASS)

Seed: 5
Target operating system: iOS
Target user behavior class: 1


In [19]:
#[CELL 2] WORKSPACE AND FILE PATHS
BASE = Path(".")
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "output"
BACKUP_DIR = BASE / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
csv_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex2_selected_users.csv"
json_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex2_summary.json"
backup_path = BACKUP_DIR / csv_out.name

print("Workspace:", BASE.resolve())
print("Dataset exists:", dataset_path.exists())

Workspace: C:\Users\anica\Lab6_File_Handling_Dantes\Lab6_File_Handling_Additional_Exercises
Dataset exists: True


In [20]:
#[CELL 3] READ DATASET AND SELECT USERS
total_records = 0
os_matches = 0
invalid_records = 0
selected = []
battery_total = 0.0
data_total = 0.0

with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    reader.fieldnames = [name.strip() for name in reader.fieldnames]

    for row in reader:
        total_records += 1

        # Requirement 1: match the assigned operating system
        if (row["Operating System"] or "").strip().lower() != TARGET_OS.lower():
            continue
        os_matches += 1

        # Skip rows with invalid or missing numbers
        try:
            behavior = int(row["User Behavior Class"])
            battery = float(row["Battery Drain (mAh/day)"])
            data_mb = float(row["Data Usage (MB/day)"])
        except (ValueError, TypeError):
            invalid_records += 1
            continue

        # Requirement 2: match the target behavior class
        if behavior != TARGET_CLASS:
            continue

        battery_total += battery
        data_total += data_mb
        selected.append({field: row[field].strip() for field in EXPORT_FIELDS})

print("Total records:", total_records)
print("Records matching OS:", os_matches)
print("Records in target class:", len(selected))
print("Invalid records skipped:", invalid_records)


Total records: 700
Records matching OS: 146
Records in target class: 22
Invalid records skipped: 0


In [21]:
#[CELL 4] CALCULATE AVERAGES
selected_count = len(selected)

if selected_count:
    avg_battery = battery_total / selected_count
    avg_data = data_total / selected_count
    print("Average battery drain (mAh/day):", round(avg_battery, 2))
    print("Average data usage (MB/day):", round(avg_data, 2))
else:
    avg_battery = None
    avg_data = None
    print("No matching records: averages are not available.")

Average battery drain (mAh/day): 467.23
Average data usage (MB/day): 201.45


In [22]:
#[CELL 5] EXPORT SELECTED USERS TO CSV
with open(csv_out, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=EXPORT_FIELDS)
    writer.writeheader()
    writer.writerows(selected)

print("Exported CSV created:", csv_out)

Exported CSV created: output\Dantes_TUPM-26-7955_ex2_selected_users.csv


In [23]:
#[CELL 6] VERIFY ROW COUNT AND WRITE JSON SUMMARY
with open(csv_out, "r", encoding="utf-8", newline="") as file:
    exported_rows = sum(1 for _ in csv.reader(file)) - 1   # minus header

count_ok = (exported_rows == selected_count)

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "operating_system": TARGET_OS,
    "user_behavior_class": TARGET_CLASS,
    "total_records": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": selected_count,
    "invalid_records": invalid_records,
    "average_battery_drain_mah_day": round(avg_battery, 2) if selected_count else None,
    "average_data_usage_mb_day": round(avg_data, 2) if selected_count else None,
    "exported_rows": exported_rows,
    "count_verification": "VERIFIED" if count_ok else "MISMATCH",
}

with open(json_out, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_out)
print("Exported rows:", exported_rows, "| Selected records:", selected_count)
print("Count verification:", "VERIFIED" if count_ok else "MISMATCH")

JSON summary created: output\Dantes_TUPM-26-7955_ex2_summary.json
Exported rows: 22 | Selected records: 22
Count verification: VERIFIED


In [24]:
#[CELL 7] BACKUP EXPORTED CSV
shutil.copy2(csv_out, backup_path)
backup_size = backup_path.stat().st_size

print("Original:", csv_out)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup size:", backup_size, "bytes")

Original: output\Dantes_TUPM-26-7955_ex2_selected_users.csv
Backup: backup\Dantes_TUPM-26-7955_ex2_selected_users.csv
Backup exists: True
Backup size: 674 bytes


In [25]:
#[CELL 8] FINAL OUTPUT
print("=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===")
print("Operating system / class :", TARGET_OS, "/", TARGET_CLASS)
print("Total records in dataset :", total_records)
print("Records matching OS      :", os_matches)
print("Records in target class  :", selected_count)
print("Average battery drain    :", round(avg_battery, 2) if selected_count else "N/A", "mAh/day")
print("Average data usage       :", round(avg_data, 2) if selected_count else "N/A", "MB/day")
print("Count verification       :", "VERIFIED" if count_ok else "MISMATCH")
print("Exported CSV file        :", csv_out)
print("JSON summary file        :", json_out)
print("Backup file and size     :", backup_path, "-", backup_size, "bytes")

=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===
Operating system / class : iOS / 1
Total records in dataset : 700
Records matching OS      : 146
Records in target class  : 22
Average battery drain    : 467.23 mAh/day
Average data usage       : 201.45 MB/day
Count verification       : VERIFIED
Exported CSV file        : output\Dantes_TUPM-26-7955_ex2_selected_users.csv
JSON summary file        : output\Dantes_TUPM-26-7955_ex2_summary.json
Backup file and size     : backup\Dantes_TUPM-26-7955_ex2_selected_users.csv - 674 bytes
